# Notebook 03 — Sales & Profitability Analytics
**SuperStore Sales & Business Performance Analytics System**

This notebook demonstrates:
- Pandas: filtering, groupBy, aggregation
- PySpark: groupBy, joins, window functions
- Key business insights extraction

In [ ]:
import sys, os

# Detect Databricks
IN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

if IN_DATABRICKS:
    import subprocess
    repo_dir = '/tmp/superstore_repo'
    if not os.path.exists(repo_dir):
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/sachintulla/Databrick-Assignment.git', repo_dir],
            check=True, capture_output=True
        )
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    MOUNT_POINT  = '/mnt/superstore'
    PROJECT_ROOT = repo_dir
    print(f'Databricks: repo cloned → {repo_dir}')
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    MOUNT_POINT = None
    print(f'Local: PROJECT_ROOT={PROJECT_ROOT}')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)
from src.analytics.sales_analytics import SalesAnalytics
from src.analytics.profitability_analytics import ProfitabilityAnalytics
from src.utils.config import load_config, get_path
cfg = load_config()
    get_path('processed_data_dir', cfg), cfg['data']['processed_file']
)
clean_df = pd.read_parquet(parquet_path)
print(f'Loaded {len(clean_df):,} rows')

## 1. KPI Summary

In [ ]:
sa = SalesAnalytics(clean_df)
pa = ProfitabilityAnalytics(clean_df)

kpis = sa.kpi_summary()
print('\n=== KPI Summary ===')
for k, v in kpis.items():
    print(f'  {k}: {v}')

## 2. Pandas: Sales Trends (Filtering + GroupBy + Aggregation)

In [ ]:
# Pandas groupby + aggregation: monthly trend
monthly = sa.monthly_sales_trend()
print('Monthly Sales Trend (first 12 rows):')
monthly.head(12)

In [ ]:
# Pandas filtering: only high-discount orders
high_discount = clean_df[clean_df['Discount'] >= 0.4]
print(f'High-discount orders (>=40%): {len(high_discount):,}')

# GroupBy + aggregation on filtered data
high_disc_by_cat = (
    high_discount
    .groupby('Category')
    .agg(
        avg_margin=('profit_margin', 'mean'),
        total_loss=('Profit', 'sum'),
        count=('Order ID', 'count')
    )
    .round(2)
)
print('\nHigh-discount impact by category:')
high_disc_by_cat

In [ ]:
# Category performance with Pandas
cat_perf = sa.category_performance()
print('Category Performance:')
cat_perf

In [ ]:
# Loss-making products — Pandas filtering + sorting
loss_products = sa.loss_making_products()
print(f'Loss-making products: {len(loss_products)}')
loss_products[['Product Name', 'Category', 'profit', 'avg_discount_pct']].head(10)

In [ ]:
# Regional analysis
regional = sa.regional_performance()
print('Regional Performance:')
regional

## 3. Profitability Deep Dive

In [ ]:
# Executive Summary
exec_sum = pa.executive_summary()
print('=== Executive Summary ===')
for k, v in exec_sum.items():
    print(f'  {k}: {v}')

In [ ]:
# Sub-category Return on Sales
ros = pa.subcategory_return_on_sales()
print('Sub-Category Return on Sales (sorted worst → best):')
ros.head(10)

In [ ]:
# Year-over-Year profit growth
yoy = pa.yoy_profit_growth()
print('YoY Profit Growth:')
yoy[['order_year', 'Category', 'profit', 'yoy_growth_pct']].dropna()

## 4. PySpark Analytics (Run on Databricks Cluster or Local Spark)

In [ ]:
# Uncomment when running on Databricks or with PySpark installed

# from pyspark.sql import SparkSession
# from src.analytics.spark_analytics import SparkAnalytics
#
# # On Databricks: spark session is already available as 'spark'
# # For local testing:
# spark = SparkSession.builder.appName('SuperStore').master('local[*]').getOrCreate()
# spark.sparkContext.setLogLevel('ERROR')
#
# sa_spark = SparkAnalytics(spark, parquet_path)
#
# # 1. KPI Summary
# print('Spark KPIs:', sa_spark.kpi_summary())
#
# # 2. Monthly trend using Spark date functions
# sa_spark.monthly_trend().show(10)
#
# # 3. Category performance using groupBy + agg
# sa_spark.category_performance().show()
#
# # 4. Join: order totals + product diversity
# sa_spark.join_order_with_product_summary().show(10)
#
# # 5. Window function: running monthly sales
# sa_spark.running_monthly_sales().show(20)
#
# # 6. Window function: product rank by category
# sa_spark.product_sales_rank_by_category().filter('rank_in_category <= 3').show(20)
#
# # 7. Loss-making products
# sa_spark.loss_making_products().show(10)

print('PySpark cells ready — uncomment to run with a Spark session.')

## 5. Key Business Insights Summary

In [ ]:
# Print a formatted business insights report
exec_sum = pa.executive_summary()
kpis = sa.kpi_summary()
risk = pa.high_discount_low_profit_risk()

print('='*65)
print('SUPERSTORE BUSINESS PERFORMANCE INSIGHTS REPORT')
print('='*65)
print(f"Total Sales     : ${kpis['total_sales']:>12,.2f}")
print(f"Total Profit    : ${kpis['total_profit']:>12,.2f}")
print(f"Profit Margin   : {kpis['avg_profit_margin_pct']:>10.1f}%")
print(f"Total Orders    : {kpis['total_orders']:>12,}")
print(f"Total Customers : {kpis['total_customers']:>12,}")
print(f"Avg Order Value : ${kpis['avg_order_value']:>12,.2f}")
print('-'*65)
print(f"Best Category   : {exec_sum['best_performing_category']}")
print(f"Worst Category  : {exec_sum['worst_performing_category']}")
print(f"Best Region     : {exec_sum['best_performing_region']}")
print(f"Worst Region    : {exec_sum['worst_performing_region']}")
print(f"Total Loss ($)  : ${exec_sum['total_loss_amount']:>12,.2f}")
print(f"Loss-Mkg SKUs   : {exec_sum['loss_making_sku_count']:>12}")
print(f"Risk Products   : {len(risk):>12} (high discount, low margin)")
print('='*65)